Emotion corpus into ChunkedCorpus format.

In [ ]:
%load_ext autoreload
%autoreload all

#import notebook helpers
from IPython.display import display,  clear_output
from tqdm import tqdm
import builtins

#import libraries
import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

#define base paths
origBasePath = os.path.abspath("./original/ktu")
print(origBasePath)

preprocBasePath = os.path.abspath("./preproc/ktu")
print(preprocBasePath)

In [ ]:
#define file paths
inFilePath = os.path.join(origBasePath, "corpus_emotions_2026-05-21.json")
print(inFilePath)

outFilePath = os.path.join(preprocBasePath, "corpus_emotions_2026-05-21-trinary.ranged.json")
print(outFilePath)

In [ ]:
#read the input

#define schema of the input
from pydantic import BaseModel, Field
from typing import List
import json

class InText(BaseModel):
    comment_id : int
    content : str    
    emotionality : float

#read the input file
inTexts : List[InText] = []

with open(inFilePath, mode="r", encoding="utf-8") as file:
    inJson = json.load(file)
    inTexts = [InText(**it) for it in inJson]

In [ ]:
#build the label set and quantization function

def quantLbl(emot: float) -> int:
	if emot <= 0.5:
		return 0 #none
	if emot <=1.5:
		return 1 #middle
	return 2 #high

#build label dictionary
labelLst = ['none', 'middle', 'high']
labelDict = { idx: lbl for idx, lbl in enumerate(labelLst) }

#
display(labelDict)

#build label groups
numHeadLabels = [len(labelDict)]
display(numHeadLabels)

In [ ]:
#build chunked corpus

from corpus.chunkedCorpus import ChunkedCorpus, ChunkedText, Chunk, ChunkSplitter

#label to assign to neutral chunks
labelNone = 0
print(f"Neutral label is: {labelNone}")

#
corpus = ChunkedCorpus(labelDefs = labelDict, labelGrps = numHeadLabels, texts = list())
for inTextIdx, inText in tqdm(enumerate(inTexts), "Input texts processed"):
    #skip empty texts, just in case
    if inText.content.strip() == 0:
        continue

    #build the labeled chunk covering all input text
    chunkLbl = quantLbl(inText.emotionality)
    chunks : List[Chunk] = [Chunk(start=0, len=len(inText.content), label=chunkLbl)]

    #build the chunked text and register with the corpus
    chunkedText = ChunkedText(id=inTextIdx, text=inText.content, chunks = chunks)
    corpus.texts.append(chunkedText)

#save results
corpus.saveToJson(outFilePath)